# Online Retail Sales Analysis — Data Cleaning & Analysis

**Project:** End-to-End Online Retail Sales Analysis using Python and Power BI

This notebook starts from the **raw Online Retail dataset** and documents the data-quality checks, cleaning decisions, feature engineering, validation, business analysis, and export workflow.

**Workflow:** Raw Data → Quality Checks → Cleaning → Feature Engineering → Validation → Analysis → Export

## 1. Import Libraries

Pandas is used for data manipulation and NumPy for numerical operations.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Load the Raw Dataset

The analysis begins with the original `Online Retail.xlsx` file, not an already-cleaned workbook.

In [2]:
file_path = "../data/Online Retail.xlsx"

raw_df = pd.read_excel(file_path)

print("Raw dataset loaded successfully.")
print("Rows:", len(raw_df))
print("Columns:", len(raw_df.columns))
raw_df.head()

Raw dataset loaded successfully.
Rows: 541909
Columns: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


## 3. Inspect the Raw Dataset

In [3]:
print("Shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())
print("\nData types:")
display(raw_df.dtypes)
print("\nSample records:")
display(raw_df.head())

Shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data types:


InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


Sample records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


## 4. Missing-Value Analysis

Identify missing values before deciding how each field should be handled.

In [4]:
missing_summary = pd.DataFrame({
    "Missing_Count": raw_df.isna().sum(),
    "Missing_Percentage": (raw_df.isna().mean() * 100).round(2)
}).sort_values("Missing_Count", ascending=False)
missing_summary

,Missing_Count,Missing_Percentage
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


## 5. Duplicate Analysis

Exact duplicate transaction rows can inflate sales and order metrics, so they are measured before removal.

In [5]:
duplicate_count = int(raw_df.duplicated().sum())
print("Duplicate rows:", duplicate_count)
print("Rows before cleaning:", len(raw_df))

Duplicate rows: 5268
Rows before cleaning: 541909


## 6. Create a Cleaning Copy

The raw dataframe remains unchanged; all transformations are performed on `clean_df`.

In [6]:
clean_df = raw_df.copy()
print("Cleaning copy shape:", clean_df.shape)

Cleaning copy shape: (541909, 8)


## 7. Standardize Data Types

In [7]:
clean_df["InvoiceDate"] = pd.to_datetime(clean_df["InvoiceDate"], errors="coerce")
clean_df["Quantity"] = pd.to_numeric(clean_df["Quantity"], errors="coerce")
clean_df["UnitPrice"] = pd.to_numeric(clean_df["UnitPrice"], errors="coerce")
clean_df["CustomerID"] = pd.to_numeric(clean_df["CustomerID"], errors="coerce")

print(clean_df.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


## 8. Remove Exact Duplicate Rows

In [8]:
before = len(clean_df)
clean_df = clean_df.drop_duplicates().copy()
print("Rows before:", before)
print("Duplicate rows removed:", before - len(clean_df))
print("Rows after:", len(clean_df))

Rows before: 541909
Duplicate rows removed: 5268
Rows after: 536641


## 9. Handle Missing Transaction Fields

Quantity and UnitPrice are required to calculate transaction value, so rows missing either field are removed.

In [9]:
before = len(clean_df)
clean_df = clean_df.dropna(subset=["Quantity", "UnitPrice"]).copy()
print("Rows removed:", before - len(clean_df))
print("Rows remaining:", len(clean_df))

Rows removed: 0
Rows remaining: 536641


## 10. Handle Missing CustomerID

For consistency with the existing Power BI dashboard, this project uses transactions with an identifiable CustomerID for its main analytical population. This is a documented **project-specific analytical decision**, not a universal retail-data rule.

In [10]:
missing_customer_before = int(clean_df["CustomerID"].isna().sum())
clean_df = clean_df.dropna(subset=["CustomerID"]).copy()
clean_df["CustomerID"] = clean_df["CustomerID"].astype(int)

print("Missing CustomerID before filtering:", missing_customer_before)
print("Rows after CustomerID filtering:", len(clean_df))
print("Missing CustomerID after filtering:", int(clean_df["CustomerID"].isna().sum()))

Missing CustomerID before filtering: 135037
Rows after CustomerID filtering: 401604
Missing CustomerID after filtering: 0


## 11. Data Quality Checks

Inspect unusual quantities, prices, descriptions, and remaining duplicates. These values are reported before further business-rule decisions.

In [11]:
quality_checks = pd.Series({
    "Negative Quantity Rows": int((clean_df["Quantity"] < 0).sum()),
    "Zero Quantity Rows": int((clean_df["Quantity"] == 0).sum()),
    "Negative UnitPrice Rows": int((clean_df["UnitPrice"] < 0).sum()),
    "Zero UnitPrice Rows": int((clean_df["UnitPrice"] == 0).sum()),
    "Missing Description": int(clean_df["Description"].isna().sum()),
    "Duplicate Rows": int(clean_df.duplicated().sum())
}, name="Count")
quality_checks

Negative Quantity Rows     8872
Zero Quantity Rows            0
Negative UnitPrice Rows       0
Zero UnitPrice Rows          40
Missing Description           0
Duplicate Rows                0
Name: Count, dtype: int64

## 12. Create SalesAmount

Transaction value is calculated as Quantity × UnitPrice.

In [12]:
clean_df["SalesAmount"] = clean_df["Quantity"] * clean_df["UnitPrice"]
clean_df[["Quantity", "UnitPrice", "SalesAmount"]].head()

,Quantity,UnitPrice,SalesAmount
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


## 13. Classify Transactions

Positive quantities represent sales, negative quantities represent returns, and zero quantities are kept as a separate category.

In [13]:
clean_df["TransactionType"] = np.select(
    [clean_df["Quantity"] > 0, clean_df["Quantity"] < 0, clean_df["Quantity"] == 0],
    ["Sale", "Return", "Zero Quantity"],
    default="Unknown"
)
clean_df["IsSale"] = (clean_df["Quantity"] > 0).astype(int)
clean_df["IsReturn"] = (clean_df["Quantity"] < 0).astype(int)
clean_df["TransactionType"].value_counts()

TransactionType
Sale      392732
Return      8872
Name: count, dtype: int64

## 14. Feature Engineering

Create calendar fields for monthly and time-based analysis.

In [14]:
clean_df["Year"] = clean_df["InvoiceDate"].dt.year
clean_df["MonthNumber"] = clean_df["InvoiceDate"].dt.month
clean_df["Month"] = clean_df["InvoiceDate"].dt.month_name()
clean_df["YearMonth"] = clean_df["InvoiceDate"].dt.to_period("M").astype(str)

clean_df[["InvoiceDate", "Year", "MonthNumber", "Month", "YearMonth"]].head()

,InvoiceDate,Year,MonthNumber,Month,YearMonth
0,2010-12-01 08:26:00,2010,12,December,2010-12
1,2010-12-01 08:26:00,2010,12,December,2010-12
2,2010-12-01 08:26:00,2010,12,December,2010-12
3,2010-12-01 08:26:00,2010,12,December,2010-12
4,2010-12-01 08:26:00,2010,12,December,2010-12


## 15. Final Validation

Confirm the cleaned dataset is consistent before analysis.

In [15]:
print("Final shape:", clean_df.shape)
print("\nRemaining missing values:")
display(clean_df.isna().sum().to_frame("Missing_Count"))
print("\nDuplicate rows:", int(clean_df.duplicated().sum()))
print("\nTransaction types:")
display(clean_df["TransactionType"].value_counts().to_frame("Rows"))

Final shape: (401604, 16)

Remaining missing values:


,Missing_Count
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,0
Country,0
SalesAmount,0
TransactionType,0



Duplicate rows: 0

Transaction types:


,Rows
TransactionType,
Sale,392732
Return,8872


## 16. Separate Sales and Returns

In [16]:
sales_df = clean_df[clean_df["Quantity"] > 0].copy()
returns_df = clean_df[clean_df["Quantity"] < 0].copy()
print("Sales rows:", len(sales_df))
print("Return rows:", len(returns_df))

Sales rows: 392732
Return rows: 8872


## 17. Core Business KPIs

In [17]:
gross_sales = sales_df["SalesAmount"].sum()
return_value = returns_df["SalesAmount"].abs().sum()
net_sales = gross_sales - return_value
total_orders = sales_df["InvoiceNo"].nunique()
total_customers = clean_df["CustomerID"].nunique()
units_sold = sales_df["Quantity"].sum()
return_rate = (len(returns_df) / len(sales_df) * 100) if len(sales_df) else 0
average_order_value = gross_sales / total_orders if total_orders else 0
average_unit_price = clean_df["UnitPrice"].mean()

kpis = pd.DataFrame([
    ["Gross Sales", gross_sales], ["Return Value", return_value], ["Net Sales", net_sales],
    ["Total Orders", total_orders], ["Total Customers", total_customers], ["Units Sold", units_sold],
    ["Return Rate (%)", return_rate], ["Average Order Value", average_order_value],
    ["Average Unit Price", average_unit_price]
], columns=["KPI", "Value"])
kpis

,KPI,Value
0,Gross Sales,"8,887,208.89"
1,Return Value,"608,689.47"
2,Net Sales,"8,278,519.42"
3,Total Orders,"18,536.00"
4,Total Customers,"4,372.00"
5,Units Sold,"5,165,886.00"
6,Return Rate (%),2.26
7,Average Order Value,479.46
8,Average Unit Price,3.47


## 18. Monthly Sales Analysis

In [18]:
monthly_sales = (sales_df.groupby("YearMonth", as_index=False)["SalesAmount"].sum()
                  .rename(columns={"SalesAmount": "GrossSales"}))
monthly_sales["YearMonth"] = pd.to_datetime(monthly_sales["YearMonth"])
monthly_sales = monthly_sales.sort_values("YearMonth")
monthly_sales

,YearMonth,GrossSales
0,2010-12-01,"570,422.73"
1,2011-01-01,"568,101.31"
2,2011-02-01,"446,084.92"
3,2011-03-01,"594,081.76"
4,2011-04-01,"468,374.33"
5,2011-05-01,"677,355.15"
6,2011-06-01,"660,046.05"
7,2011-07-01,"598,962.90"
8,2011-08-01,"644,051.04"
9,2011-09-01,"950,690.20"


## 19. Country Analysis

In [19]:
country_analysis = (clean_df.groupby("Country", as_index=False)
    .agg(NetSales=("SalesAmount", "sum"), Orders=("InvoiceNo", "nunique"), Customers=("CustomerID", "nunique"))
    .sort_values("NetSales", ascending=False))
country_analysis.head(20)

,Country,NetSales,Orders,Customers
35,United Kingdom,"6,747,156.15",19857,3950
23,Netherlands,"284,661.54",101,9
10,EIRE,"250,001.78",319,3
14,Germany,"221,509.47",603,95
13,France,"196,626.05",458,87
0,Australia,"137,009.77",69,9
32,Switzerland,"55,739.40",71,21
30,Spain,"54,756.03",105,31
3,Belgium,"40,910.96",119,25
31,Sweden,"36,585.41",46,8


## 20. Product Analysis

In [20]:
product_analysis = (clean_df.groupby(["StockCode", "Description"], dropna=False, as_index=False)
    .agg(NetSales=("SalesAmount", "sum"), UnitsSold=("Quantity", lambda x: x[x > 0].sum()),
         AverageUnitPrice=("UnitPrice", "mean"), Orders=("InvoiceNo", "nunique"))
    .sort_values("NetSales", ascending=False))
product_analysis.head(20)

,StockCode,Description,NetSales,UnitsSold,AverageUnitPrice,Orders
1249,22423,REGENCY CAKESTAND 3 TIER,"132,567.70",12384,12.43,1884
3593,85123A,WHITE HANGING HEART T-LIGHT HOLDER,"93,767.80",36706,2.89,2013
3586,85099B,JUMBO BAG RED RETROSPOT,"83,056.52",46078,2.01,1643
2613,47566,PARTY BUNTING,"67,628.43",15283,4.87,1399
3915,POST,POSTAGE,"66,710.24",3120,37.89,1194
2818,84879,ASSORTED COLOUR BIRD ORNAMENT,"56,331.91",35263,1.68,1385
1937,23084,RABBIT NIGHT LIGHT,"51,042.84",27153,2.01,816
2680,79321,CHILLI LIGHTS,"45,915.41",9646,5.42,525
932,22086,PAPER CHAIN KIT 50'S CHRISTMAS,"41,423.78",15591,2.93,990
1324,22502,PICNIC BASKET WICKER 60 PIECES,"39,619.50",61,649.50,2


## 21. Customer Analysis

In [21]:
customer_analysis = (clean_df.groupby("CustomerID", as_index=False)
    .agg(NetSales=("SalesAmount", "sum"), Orders=("InvoiceNo", "nunique"),
         UnitsPurchased=("Quantity", lambda x: x[x > 0].sum()))
    .sort_values("NetSales", ascending=False))
customer_analysis.head(20)

,CustomerID,NetSales,Orders,UnitsPurchased
1703,14646,"279,489.02",77,197491
4233,18102,"256,438.49",62,64124
3758,17450,"187,322.17",55,69973
1895,14911,"132,458.73",248,80490
55,12415,"123,725.45",26,77670
1345,14156,"113,214.59",66,57768
3801,17511,"88,125.38",46,64549
3202,16684,"65,892.08",31,50255
1005,13694,"62,690.54",60,63312
2192,15311,"59,284.19",118,38147


## 22. Returns Analysis

In [22]:
returns_analysis = (returns_df.groupby("Country", as_index=False)
    .agg(ReturnValue=("SalesAmount", lambda x: x.abs().sum()),
         ReturnTransactions=("InvoiceNo", "nunique"), ReturnedUnits=("Quantity", lambda x: x.abs().sum()))
    .sort_values("ReturnValue", ascending=False))
returns_analysis.head(20)

,Country,ReturnValue,ReturnTransactions,ReturnedUnits
27,United Kingdom,"537,868.49",3208,259167
7,EIRE,"15,260.68",59,4196
10,France,"12,308.26",69,1623
22,Singapore,"12,158.90",3,7
11,Germany,"7,168.93",146,1815
23,Spain,"6,802.53",15,1127
20,Portugal,"4,380.08",13,78
15,Japan,"2,075.75",9,798
26,USA,"1,849.47",2,1424
24,Sweden,"1,782.42",10,446


## 23. Quantity Sign Summary

In [23]:
quantity_summary = pd.DataFrame([
    ["Positive Sales", int((clean_df["Quantity"] > 0).sum()), clean_df.loc[clean_df["Quantity"] > 0, "SalesAmount"].sum()],
    ["Negative Returns", int((clean_df["Quantity"] < 0).sum()), clean_df.loc[clean_df["Quantity"] < 0, "SalesAmount"].sum()],
    ["Zero Quantity", int((clean_df["Quantity"] == 0).sum()), clean_df.loc[clean_df["Quantity"] == 0, "SalesAmount"].sum()]
], columns=["TransactionType", "Rows", "SalesAmount"])
quantity_summary

,TransactionType,Rows,SalesAmount
0,Positive Sales,392732,"8,887,208.89"
1,Negative Returns,8872,"-608,689.47"
2,Zero Quantity,0,0.00


## 24. Final Clean Dataset Preview

In [24]:
clean_df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SalesAmount,TransactionType,IsSale,IsReturn,Year,MonthNumber,Month,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,Sale,1,0,2010,12,December,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale,1,0,2010,12,December,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,Sale,1,0,2010,12,December,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale,1,0,2010,12,December,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,Sale,1,0,2010,12,December,2010-12
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,15.30,Sale,1,0,2010,12,December,2010-12
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,25.50,Sale,1,0,2010,12,December,2010-12
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,11.10,Sale,1,0,2010,12,December,2010-12
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,11.10,Sale,1,0,2010,12,December,2010-12
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom,54.08,Sale,1,0,2010,12,December,2010-12


## 25. Export the Project-Ready Workbook

Export the cleaned dataset and supporting analysis tables for the Power BI project.

In [25]:
output_file = "../data/Online_Retail_Analysis_PowerBI_Final.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    clean_df.to_excel(writer, sheet_name="Clean_Data", index=False)
    monthly_sales.to_excel(writer, sheet_name="Monthly_Sales", index=False)
    country_analysis.to_excel(writer, sheet_name="Country_Analysis", index=False)
    product_analysis.to_excel(writer, sheet_name="Product_Analysis", index=False)
    customer_analysis.to_excel(writer, sheet_name="Customer_Analysis", index=False)
    returns_analysis.to_excel(writer, sheet_name="Returns_Analysis", index=False)
    kpis.to_excel(writer, sheet_name="Final_KPIs", index=False)
    quantity_summary.to_excel(writer, sheet_name="Quantity_Summary", index=False)

print("Export completed successfully:", output_file)

Export completed successfully: ../data/Online_Retail_Analysis_PowerBI_Final.xlsx


## Conclusion

This notebook documents the project from the raw dataset through data-quality assessment, cleaning, feature engineering, validation, business analysis, and export. The CustomerID filtering decision is explicitly documented because it is used to reproduce the analytical population of the existing Power BI dashboard.

In [ ]:
output_file = "../data/Online_Retail_Analysis_PowerBI_Final.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    clean_df.to_excel(writer, sheet_name="Clean_Data", index=False)
    monthly_sales.to_excel(writer, sheet_name="Monthly_Sales", index=False)
    country_analysis.to_excel(writer, sheet_name="Country_Analysis", index=False)
    product_analysis.to_excel(writer, sheet_name="Product_Analysis", index=False)
    customer_analysis.to_excel(writer, sheet_name="Customer_Analysis", index=False)
    returns_analysis.to_excel(writer, sheet_name="Returns_Analysis", index=False)
    kpis.to_excel(writer, sheet_name="Final_KPIs", index=False)
    quantity_summary.to_excel(writer, sheet_name="Quantity_Summary", index=False)

print("Export completed successfully:", output_file)

Export completed successfully: ../data/Online_Retail_Analysis_PowerBI_Final.xlsx
